# Rangkuman Kinerja Proyek NIDS Lintas-Jaringan (SFM + XGBoost)

**Untuk presentasi / PPT.** Notebook ini merangkum seluruh hasil eksperimen nyata proyek:
dari dua dataset sumber, pembersihan data, pemetaan fitur (SFM), pelatihan & pengujian model,
adaptasi domain, efisiensi *edge*, hingga validasi trafik nyata (FAR) di AWS.

> **Prinsip kejujuran data:** semua angka berasal dari eksperimen nyata (berkas `*.json` di folder induk
> dan hasil capture AWS). Bila berkas JSON tersedia, notebook memuatnya; bila tidak, dipakai nilai
> *fallback* yang identik dengan hasil tercatat sehingga notebook tetap jalan di mana pun (mis. SageMaker).

Jalankan sel berurutan dari atas ke bawah.

## 0. Setup & pemuatan hasil

Jalankan sel instalasi di bawah **sekali** bila kernel belum punya paket (mis. error
`No module named 'matplotlib'`). Setelah instalasi selesai, lanjutkan ke sel berikutnya
(tak perlu restart untuk `%pip install`).

In [ ]:
# Instalasi paket bila belum ada (aman dijalankan berulang).
import importlib, sys, subprocess
need = [m for m in ('matplotlib', 'pandas', 'numpy') if importlib.util.find_spec(m) is None]
if need:
    print('Menginstal:', need)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *need], check=True)
    print('Selesai. Bila import di sel berikutnya masih gagal, Restart Kernel lalu jalankan lagi.')
else:
    print('Semua paket sudah tersedia.')

In [ ]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.dpi': 110, 'font.size': 11, 'axes.grid': True, 'grid.alpha': 0.3})

# Cari folder berkas hasil JSON (folder induk dari notebooks/, atau folder saat ini)
CANDIDATES = ['..', '.', '../unswnb-15', 'unswnb-15']
DATA_DIR = next((d for d in CANDIDATES if os.path.exists(os.path.join(d, 'cross_dataset_baseline.json'))), '..')
print('DATA_DIR =', os.path.abspath(DATA_DIR))

def load_json(name, fallback=None):
    p = os.path.join(DATA_DIR, name)
    if os.path.exists(p):
        with open(p) as f:
            print('  loaded:', name)
            return json.load(f)
    print('  (fallback):', name)
    return fallback

## 1. Dua Dataset Sumber

Proyek ini sengaja memakai **pasangan dataset dari sumber & alat ekstraksi berbeda** untuk menguji
generalisasi lintas-jaringan secara jujur:

- **CSE-CIC-IDS2018** — trafik *testbed* CIC (2018), diekstraksi dengan **CICFlowMeter**.
- **UNS-NB15** (UNSW-NB15) — trafik dibangkitkan IXIA PerfectStorm (2015), diekstraksi dengan **Argus + Bro/Zeek**.

Perbedaan sumber & ekstraktor ini bukan kelemahan, melainkan prasyarat menguji *cross-network robustness*.

In [ ]:
inv = load_json('feature_inventory.json', fallback={
    'cic_ids2018': {'n_features': 68, 'extractor': 'CICFlowMeter'},
    'unsw_nb15':   {'n_features': 42, 'extractor': 'Argus + Bro/Zeek (+12 custom algorithms)'}
})

# Tabel perbandingan dua dataset (angka nyata dari proyek)
compare = pd.DataFrame([
    {'Aspek': 'Tahun / sumber',        'CIC (CSE-CIC-IDS2018)': '2018, testbed CIC',       'UNS (UNSW-NB15)': '2015, IXIA PerfectStorm'},
    {'Aspek': 'Alat ekstraksi',        'CIC (CSE-CIC-IDS2018)': inv['cic_ids2018']['extractor'], 'UNS (UNSW-NB15)': inv['unsw_nb15']['extractor']},
    {'Aspek': 'Jumlah fitur',          'CIC (CSE-CIC-IDS2018)': inv['cic_ids2018']['n_features'], 'UNS (UNSW-NB15)': inv['unsw_nb15']['n_features']},
    {'Aspek': 'Skema label',           'CIC (CSE-CIC-IDS2018)': 'Benign + 14 jenis serangan', 'UNS (UNSW-NB15)': 'Normal + 9 jenis serangan'},
    {'Aspek': 'Record dipakai (biner)','CIC (CSE-CIC-IDS2018)': '1.348.453 normal / 274.808 attack', 'UNS (UNSW-NB15)': '175.341 latih / 82.332 uji'},
    {'Aspek': 'Granularitas',          'CIC (CSE-CIC-IDS2018)': 'per-flow', 'UNS (UNSW-NB15)': 'per-flow'},
])
compare

In [ ]:
# Jenis serangan di kedua dataset (untuk konteks slide)
cic_attacks = ['Benign', 'DoS (Hulk/GoldenEye/Slowloris/SlowHTTPTest)', 'DDoS (LOIC/HOIC)',
               'Brute-Force (FTP/SSH)', 'Web (XSS/SQLi/Brute)', 'Infiltration', 'Botnet (Ares)']
unsw_attacks = ['Normal', 'Generic', 'Exploits', 'Fuzzers', 'DoS', 'Reconnaissance',
                'Analysis', 'Backdoor', 'Shellcode', 'Worms']
print('CSE-CIC-IDS2018 (kelompok serangan):')
for a in cic_attacks: print('   -', a)
print('\nUNS-NB15 / UNSW-NB15 (10 kelas):')
for a in unsw_attacks: print('   -', a)
print('\nUntuk tahap ini keduanya dibinerkan: attack vs normal.')

In [ ]:
# Diagram batang: jumlah fitur per dataset
fig, ax = plt.subplots(figsize=(5.5, 3.2))
names = ['CIC\n(68, CICFlowMeter)', 'UNS\n(42, Argus+Bro/Zeek)']
vals = [inv['cic_ids2018']['n_features'], inv['unsw_nb15']['n_features']]
bars = ax.bar(names, vals, color=['#4C72B0', '#DD8452'])
ax.set_ylabel('Jumlah fitur')
ax.set_title('Perbedaan jumlah fitur antar-ekstraktor')
for b, v in zip(bars, vals):
    ax.text(b.get_x()+b.get_width()/2, v+1, str(v), ha='center', fontweight='bold')
plt.tight_layout(); plt.show()

## 2. Pembersihan Data & Binerisasi

Langkah pra-pemrosesan utama:
1. **Buang baris rusak** (inf / NaN dari pembagian, mis. `Flow Byts/s` saat durasi 0).
2. **Binerisasi label**: semua jenis serangan -> `attack` (1), sisanya `normal` (0).
3. **Verifikasi label** pada CIC: `Benign`=0 menghasilkan tepat 1.348.453 normal & 274.808 attack.
4. **StandardScaler (z-score) per-dataset terpisah** agar perbedaan satuan/skala ternormalisasi
   tanpa mengarang konversi antar-alat.

In [ ]:
# Komposisi kelas biner kedua dataset (angka verifikasi nyata)
# CIC: total populasi biner terverifikasi. UNSW: diturunkan dari confusion same_unsw (test set).
cic_normal, cic_attack = 1348453, 274808
try:
    cm = base['results']['model_A']['same_unsw']['confusion']  # [[TN,FP],[FN,TP]] pd test UNSW
    unsw_normal = cm[0][0] + cm[0][1]
    unsw_attack = cm[1][0] + cm[1][1]
except Exception:
    unsw_normal, unsw_attack = 37000, 45332  # fallback (test set UNSW)

fig, axes = plt.subplots(1, 2, figsize=(9.2, 4.4))
for ax, (title, nrm, atk) in zip(axes, [
        (f'CSE-CIC-IDS2018\n(total {cic_normal+cic_attack:,} flow)', cic_normal, cic_attack),
        (f'UNS (test set)\n(total {unsw_normal+unsw_attack:,} flow)', unsw_normal, unsw_attack)]):
    ax.pie([nrm, atk], labels=['normal', 'attack'], autopct='%1.1f%%',
           colors=['#55A868', '#C44E52'], startangle=90, explode=(0, 0.05))
    ax.set_title(title)
plt.suptitle('Komposisi kelas pasca-binerisasi', fontsize=12)
plt.tight_layout(); plt.show()
print('Rasio ketidakseimbangan (normal:attack):')
print('  CIC  = %.1f : 1  (mayoritas normal)' % (cic_normal/cic_attack))
print('  UNS  = 1 : %.1f  (mayoritas attack -> distribusi berbeda dari CIC)' % (unsw_attack/unsw_normal))

## 3. Pelatihan & Pengujian OFFLINE — Klasifikasi BINER

Model: **XGBoost** (`max_depth=8, lr=0.1, n_estimators=200, subsample/colsample=0.8, binary:logistic`).
Metrik utama **MCC** (tahan *class imbalance*).

**Cara & skenario pengujian (biner, offline):**
- **Label:** semua jenis serangan digabung -> `attack` (1); sisanya `normal` (0).
- **Pra-pemrosesan:** `StandardScaler` (z-score) **di-fit hanya pada data latih** (no leakage);
  diterapkan **per-dataset terpisah** agar perbedaan satuan/skala ternormalkan.
- **Empat skenario** (mengukur generalisasi lintas-jaringan):
  1. **Same-CIC** — latih & uji pada CIC (split internal). *In-domain.*
  2. **Same-UNS** — latih & uji pada UNSW. *In-domain.*
  3. **CIC→UNS** — latih di CIC, **uji di UNSW** (jaringan asing). *Cross-network.*
  4. **UNS→CIC** — latih di UNSW, **uji di CIC**. *Cross-network.*
- **Interpretasi:** *generalization gap* = MCC in-domain − MCC cross-network. Gap besar =
  model tak transfer antar-jaringan (inti masalah paper).

> Catatan: fitur bersama yang dipakai melatih model ini (9 fitur SFM) baru **dijelaskan di
> Bagian 4** — sengaja: tunjukkan **kegagalan** dulu (di sini), baru **fondasi/solusi** (SFM).
> Notebook sumber: `03_cross_dataset_baseline.ipynb`. **Multi-class** dibahas terpisah di Bagian 8
> (setelah seluruh alur biner selesai).

In [ ]:
base = load_json('cross_dataset_baseline.json', fallback={'results': {'model_A': {
    'same_cic': {'mcc': 0.9134, 'f1': 0.9277}, 'same_unsw': {'mcc': 0.7448, 'f1': 0.8904},
    'cic2unsw': {'mcc': -0.0719, 'f1': 0.0224}, 'unsw2cic': {'mcc': -0.0613, 'f1': 0.0088}}}})
rA = base['results']['model_A']
scen = ['Same-CIC', 'Same-UNS', 'CIC->UNS', 'UNS->CIC']
keys = ['same_cic', 'same_unsw', 'cic2unsw', 'unsw2cic']
mccs = [rA[k]['mcc'] for k in keys]
f1s  = [rA[k].get('f1', float('nan')) for k in keys]

# Tabel ringkas MCC + F1
tbl = pd.DataFrame({'Skenario': scen, 'MCC': np.round(mccs, 4), 'F1': np.round(f1s, 4)})
display(tbl)

# Grouped bar: MCC vs F1 per skenario
x = np.arange(len(scen)); ww = 0.38
fig, ax = plt.subplots(figsize=(7.2, 3.8))
b1 = ax.bar(x-ww/2, mccs, ww, label='MCC', color='#4C72B0')
b2 = ax.bar(x+ww/2, f1s,  ww, label='F1',  color='#DD8452')
ax.axhline(0, color='k', lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(scen)
ax.set_ylabel('Skor'); ax.set_ylim(-0.2, 1.0)
ax.set_title('Model A: in-domain tinggi, lintas-jaringan runtuh (~0)')
for bars in (b1, b2):
    for b in bars:
        v = b.get_height()
        ax.text(b.get_x()+b.get_width()/2, v + (0.02 if v>=0 else -0.07), f'{v:.2f}', ha='center', fontsize=8)
ax.legend(); plt.tight_layout(); plt.show()
print('Baik MCC maupun F1 runtuh lintas-jaringan; generalization gap MCC ~0.81-0.99.')
print('Catatan: F1 lintas-jaringan sangat rendah (mendekati 0) -> model gagal mengenali kelas attack di jaringan asing.')

## 4. Semantic Feature Mapping (SFM) & Validasi — *Bagaimana kedua dataset bisa dilatih bersama*

> **Transisi (masalah → solusi):** Bagian 3 menunjukkan model **runtuh lintas-jaringan**. Muncul
> pertanyaan mendasar: CIC (68 fitur, CICFlowMeter) dan UNSW (42 fitur, Argus/Bro) punya nama &
> jumlah kolom berbeda — **bagaimana keduanya bahkan bisa dilatih/diuji dengan fitur yang sama?**
> Jawabannya = **SFM** di bawah ini: fondasi yang memungkinkan pengujian cross-dataset (Bagian 3),
> sekaligus titik awal solusi (Bagian 5: joint training, few-shot).

SFM memetakan fitur berfungsi-sama antar-dataset meski nama kolomnya berbeda
(mis. `Flow Duration` <-> `dur`). Tiap pasangan divalidasi statistik (rentang, distribusi, satuan),
**bukan** sekadar kemiripan nama. Verdict:
- `aligned` — langsung sepadan.
- `scale-mismatch` — sepadan tapi perlu penskalaan (ditangani z-score per-dataset).
- `likely-different-feature` — **dibuang** (mis. TCP window & IAT), studi kasus *feature-extractor mismatch*.

**Berkas & pipeline penghasil `mapping_validation.json`:**
SFM tahap ini adalah **analisis offline dari dataset yang sudah ada** — **TIDAK memakai
capture pcap / berkas `.sh`**. (Capture pcap via skrip `.sh` seperti `aws/capture_target.sh`
baru dipakai pada tahap terpisah: **validasi trafik nyata AWS** di Bagian 7, bukan di SFM.)

| Item | Berkas |
|---|---|
| Notebook penghasil | `notebooks/02_mapping_validation.ipynb` (dijalankan di SageMaker) |
| Input CIC | `CICDDoS2018/data/cleaned_100.pkl` (fitur ter-`StandardScaler`; di-*un-scale* dulu agar adil) |
| Input UNSW | `unswnb-15/data/UNSW_NB15_testing-set.csv` (175.341 record = data latih menurut jumlah) |
| Inventaris fitur | `notebooks/01_feature_inventory.ipynb` → `feature_inventory.json` (68 fitur CIC, 42 fitur UNSW) |
| Output | `mapping_validation.json` + `mapping_validation.csv` (statistik per pasangan + verdict) |

Metode validasi (di notebook 02): un-scale CIC ke satuan asli → bandingkan **median & p99**
tiap pasangan fitur → rasio p99 & jarak orde-magnitudo $|\log_{10} r|$ menentukan verdict
(`aligned`/`scale-mismatch`/`likely-different`), dikonfirmasi penalaran domain (satuan/definisi).

In [ ]:
mapv = load_json('mapping_validation.json', fallback=[])
if mapv:
    dfm = pd.DataFrame(mapv)[['cic', 'unsw', 'hyp', 'verdict']]
    dfm.columns = ['Fitur CIC', 'Fitur UNS', 'Hipotesis', 'Verdict']
    display(dfm)
    print('\nRingkasan verdict:')
    print(pd.Series([m['verdict'] for m in mapv]).value_counts().to_string())
else:
    print('mapping_validation.json tidak ditemukan.')

**Himpunan fitur final (Model A, 9 fitur irisan kuat):**
`duration, fwd_pkts, bwd_pkts, fwd_bytes, bwd_bytes, fwd_mean, bwd_mean, src_load, dst_load`.
Model B menambah `fwd_iat, bwd_iat` (11 fitur) — setara Model A namun kurang ringkas.

Diagram Venn di bawah memvisualkan hasil SFM: dari 68 fitur CIC dan 42 fitur UNS,
hanya **9 fitur** yang benar-benar sepadan (irisan) dan dipakai bersama sebagai Model A.

In [ ]:
# Diagram Venn irisan fitur CIC vs UNS. Irisan = 9 fitur SFM (Model A).
N_CIC, N_UNS, N_SHARED = 68, 42, 9
shared_feats = ['duration','fwd_pkts','bwd_pkts','fwd_bytes','bwd_bytes','fwd_mean','bwd_mean','src_load','dst_load']
try:
    from matplotlib_venn import venn2
    fig, ax = plt.subplots(figsize=(6.4, 4.4))
    v = venn2(subsets=(N_CIC-N_SHARED, N_UNS-N_SHARED, N_SHARED),
              set_labels=('CIC\n(68 fitur,\nCICFlowMeter)', 'UNS\n(42 fitur,\nArgus+Bro)'), ax=ax)
    if v.get_label_by_id('10'): v.get_label_by_id('10').set_text(f'{N_CIC-N_SHARED}\nunik')
    if v.get_label_by_id('01'): v.get_label_by_id('01').set_text(f'{N_UNS-N_SHARED}\nunik')
    if v.get_label_by_id('11'): v.get_label_by_id('11').set_text(f'{N_SHARED} fitur SFM\n(Model A)')
    ax.set_title('SFM: irisan fitur CIC \u2229 UNS = 9 fitur sepadan (tervalidasi statistik)')
    plt.tight_layout(); plt.show()
except Exception:
    from matplotlib.patches import Circle
    fig, ax = plt.subplots(figsize=(6.8, 4.4)); ax.set_aspect('equal'); ax.axis('off')
    ax.add_patch(Circle((0.38, 0.5), 0.34, alpha=0.45, color='#4C72B0'))
    ax.add_patch(Circle((0.62, 0.5), 0.30, alpha=0.45, color='#DD8452'))
    ax.text(0.17, 0.5, 'CIC\n68 fitur\n(CICFlowMeter)', ha='center', va='center', fontsize=10)
    ax.text(0.83, 0.5, 'UNS\n42 fitur\n(Argus+Bro)', ha='center', va='center', fontsize=10)
    ax.text(0.50, 0.5, f'{N_SHARED} fitur\nSFM', ha='center', va='center', fontsize=10, fontweight='bold')
    ax.set_xlim(0,1); ax.set_ylim(0.1,0.9)
    ax.set_title('SFM: irisan fitur CIC \u2229 UNS = 9 fitur SFM (Model A)')
    plt.tight_layout(); plt.show()
    print('(matplotlib_venn tidak ada -> diagram manual. Install: pip install matplotlib-venn)')

print('9 fitur SFM (irisan):', ', '.join(shared_feats))
print('Catatan: 2 fitur IAT (fwd_iat,bwd_iat) utk Model B; 2 fitur TCP window DIBUANG (mismatch).')

## 5. Diagnosis & Solusi: Distribution Shift, Bukan Kekurangan Fitur

- **Joint training** (gabung CIC+UNS) mencapai MCC hampir setara in-domain di kedua jaringan
  serentak -> membuktikan **SFM valid** (9 fitur cukup ekspresif). Maka celah = *distribution shift*.
- **Few-shot 1% label target** memulihkan MCC dari negatif ke 0.65-0.90.
- **Mixup** (tanpa label target) juga memulihkan sebagian besar.

> Irisan fitur (SFM, 9 fitur) sudah divisualkan di Bagian 3. Di sini fokusnya: meski fitur
> sudah sepadan, model single-source tetap runtuh -> penyebabnya *distribution shift*, dan
> kalibrasi domain (few-shot/mixup) yang memperbaikinya.

In [ ]:
da = load_json('domain_adaptation.json', fallback=None)
align = load_json('cross_network_alignment.json', fallback=None)

if da:
    fs_c = pd.DataFrame(da['fewshot']['cic2unsw'])
    fs_u = pd.DataFrame(da['fewshot']['unsw2cic'])

    def fmt(df):
        d = df.copy()
        d['frac'] = (d['frac']*100).map(lambda v: f'{v:g}%')
        cols = [c for c in ['frac','n_target','mcc','f1','acc'] if c in d.columns]
        d = d[cols].round(4)
        d.columns = ['Fraksi label target','n_flow target','MCC','F1','Akurasi'][:len(cols)]
        return d

    # DUA TABEL TERPISAH karena kedua arah berbeda hasil (asimetris)
    print('=== Tabel 1: CIC -> UNS (latih CIC + x% UNS, uji UNS) ===')
    display(fmt(fs_c))
    print('\n=== Tabel 2: UNS -> CIC (latih UNS + x% CIC, uji CIC) ===')
    display(fmt(fs_u))

    # Kurva perbandingan kedua arah
    fig, ax = plt.subplots(figsize=(6.6, 3.8))
    ax.plot(fs_c['frac']*100, fs_c['mcc'], 'o-', label='CIC->UNS', color='#4C72B0')
    ax.plot(fs_u['frac']*100, fs_u['mcc'], 's-', label='UNS->CIC', color='#DD8452')
    ax.axhline(0, color='k', lw=0.8, ls=':')
    ax.set_xlabel('Fraksi label target (%)'); ax.set_ylabel('MCC lintas-jaringan')
    ax.set_title('Few-shot: 1% label target sudah memulihkan MCC (asimetris antar-arah)')
    ax.legend(); plt.tight_layout(); plt.show()

    print('Asimetri: UNS->CIC pulih jauh lebih tinggi (0%%=%.3f -> 1%%=%.3f, ~in-domain)'
          % (fs_u.iloc[0]['mcc'], fs_u.iloc[1]['mcc']))
    print('          CIC->UNSW pulih lebih rendah      (0%%=%.3f -> 1%%=%.3f)'
          % (fs_c.iloc[0]['mcc'], fs_c.iloc[1]['mcc']))
    print('Mixup (tanpa label target): CIC->UNS MCC=%.3f ; UNS->CIC MCC=%.3f' %
          (da['mixup']['cic2unsw']['mcc'], da['mixup']['unsw2cic']['mcc']))
else:
    print('domain_adaptation.json tidak ditemukan.')

In [ ]:
# Tabel strategi penyelarasan (baseline / CORAL / few-shot / mixup / joint).
# Kolom disamakan berdasarkan 'diuji di jaringan mana', BUKAN arah transfer,
# agar baris joint (dilatih di keduanya) tidak menyesatkan.
align = align if 'align' in dir() else load_json('cross_network_alignment.json', fallback=None)
da = da if 'da' in dir() else load_json('domain_adaptation.json', fallback=None)
if align:
    cs = {r['strategi']: r for r in align['cross_summary']}
    def get(strat, direction):
        return cs.get(strat, {}).get(direction, float('nan'))
    rows = [
        {'Strategi': 'Baseline single-source (0% target)',
         'MCC di test UNS': get('Baseline single-source', 'cic2unsw'),
         'MCC di test CIC':  get('Baseline single-source', 'unsw2cic')},
        {'Strategi': 'CORAL alignment',
         'MCC di test UNS': get('CORAL alignment', 'cic2unsw'),
         'MCC di test CIC':  get('CORAL alignment', 'unsw2cic')},
    ]
    # Sisipkan baris few-shot (1/5/10/25%) dari domain_adaptation.json bila tersedia.
    # MCC di test UNSW <- arah cic2unsw ; MCC di test CIC <- arah unsw2cic.
    if da:
        fc = {round(r['frac'], 4): r['mcc'] for r in da['fewshot']['cic2unsw']}
        fu = {round(r['frac'], 4): r['mcc'] for r in da['fewshot']['unsw2cic']}
        for frac in [0.01, 0.05, 0.10, 0.25]:
            rows.append({'Strategi': f'Few-shot {int(frac*100)}% label target',
                         'MCC di test UNS': fc.get(frac, float('nan')),
                         'MCC di test CIC':  fu.get(frac, float('nan'))})
        rows.append({'Strategi': 'Mixup (tanpa label target)',
                     'MCC di test UNS': da['mixup']['cic2unsw']['mcc'],
                     'MCC di test CIC':  da['mixup']['unsw2cic']['mcc']})
    rows.append({'Strategi': 'Joint training (latih di keduanya, 100%)',
                 'MCC di test UNS': align['joint']['unsw_test_mcc'],
                 'MCC di test CIC':  align['joint']['cic_test_mcc']})
    tbl = pd.DataFrame(rows).round(4)
    display(tbl)
    print('Kolom = performa pada test set jaringan tsb (bukan arah transfer).')
    print('Alur pemulihan: baseline (~0) -> few-shot 1% (lompat) -> mendatar -> joint (batas atas).')
else:
    print('cross_network_alignment.json tidak ditemukan.')

### 5b. Verifikasi kuantitatif: Jarak Wasserstein turun setelah kalibrasi

In [ ]:
w = load_json('wasserstein_shift.json', fallback=None)
if w:
    def row(direction):
        r = w['results'][direction]
        return {'Arah': direction,
                'Baseline': r['before']['mean'],
                'Few-shot 1%': r['fewshot_1pct']['mean'],
                'Mixup': r['mixup']['mean'],
                'Batas-bawah': r['target_train_lb']['mean']}
    dw = pd.DataFrame([row('cic2unsw'), row('unsw2cic')])
    display(dw)
    print('W1 (rata-rata 9 fitur) mengecil ke arah target -> kalibrasi benar menggeser distribusi.')
else:
    print('wasserstein_shift.json tidak ditemukan.')

## 6. Efisiensi Model (Edge / Green AI)

Diukur nyata pada 1 vCPU (*single-thread*, batch=1) untuk meniru penyebaran *edge*.

In [ ]:
eff = load_json('model_efficiency.json', fallback={
    'size_kb': 2961.8, 'latency_per_flow_us': {'mean': 439.3, 'median': 434.2, 'std': 22.9},
    'throughput_flows_per_sec': {'incremental': 2276}})
eff_tbl = pd.DataFrame([
    {'Metrik': 'Ukuran model biner (9 fitur, 200 pohon)', 'Nilai': f"{eff['size_kb']/1024:.1f} MB"},
    {'Metrik': 'Latensi inferensi per flow (median, 1 vCPU)', 'Nilai': f"{eff['latency_per_flow_us']['median']:.0f} us"},
    {'Metrik': 'Latensi inferensi per flow (mean +/- std)', 'Nilai': f"{eff['latency_per_flow_us']['mean']:.0f} +/- {eff['latency_per_flow_us']['std']:.0f} us"},
    {'Metrik': 'Throughput inkremental (1 vCPU)', 'Nilai': f"~{eff['throughput_flows_per_sec']['incremental']:,} flow/detik"},
])
eff_tbl

## 7. Validasi Trafik Nyata di AWS — False Alarm Rate (FAR)

Pipeline: **tcpdump (pcap) -> NFStream (9 fitur SFM) -> XGBoost**. Fase 1 = trafik *benign* saja,
sehingga tiap prediksi `attack` = alarm palsu. `FAR = false_alarm / total_flow`.

**Cara & skenario pengujian di AWS (arsitektur & dua fase):**
- **Infra:** 2 EC2 di VPC privat — **Attacker** + **Target+Analyzer** (gabungan). Capture pcap
  (`aws/capture_target.sh`) & inferensi (`aws/unsw_extract_infer.py`) di mesin yang sama; akses via SSM.
- **Fase 1 — FAR (tanpa serangan):** hanya trafik benign dibangkitkan. Diuji **ramp bertahap**
  S0 (3 mnt) -> D1 (1 jam) -> D2 (6 jam) -> D3 (24 jam), tiap tahap punya gate (mis. |z|<=6, FAR!=1).
  Tujuan: buktikan FAR rendah **stabil lintas-durasi** (bukan artefak cuplikan pendek).
- **Fase 2 — Deteksi (dengan serangan):** Attacker melancarkan serangan **terjadwal** (satu jenis
  per selang waktu + cooldown), sehingga ground-truth diketahui dari **timeline**. Ini memungkinkan
  evaluasi **biner** (Bagian 7b) MAUPUN **multi-class** (Bagian 8b), sebab tiap flow bisa diberi
  label kategori dari waktunya.
- **Satuan (audit):** fitur `duration` = **mikrodetik** agar cocok scaler CIC; pembagi laju = detik.

> Notebook/skrip: `aws/runbook.md` (operasi), `unsw_extract_infer.py` (FAR/deteksi biner),
> `25_multiclass_aws.ipynb` (multi-class AWS). Bagian ini (7) memuat hasil **FAR**;
> notebook mencoba memuat `far_log.jsonl` (dari mesin AWS / S3), bila tak ada dipakai hasil tercatat.

In [ ]:
# Coba muat far_log.jsonl (real-traffic). Cari di beberapa lokasi umum.
far_paths = ['/opt/unsw/results/far_log.jsonl', os.path.join(DATA_DIR, 'far_log.jsonl'), 'far_log.jsonl']
far_rows = None
for p in far_paths:
    if os.path.exists(p):
        far_rows = [json.loads(l) for l in open(p) if l.strip()]
        print('loaded far_log.jsonl dari', p); break

if not far_rows:
    print('(fallback) memakai hasil tercatat S0 & D1.')
    far_rows = [
        {'pcap': 'ramp_s0 (3 menit)', 'n_flow': 203,  'n_false_alarm': 0,  'far': 0.0},
        {'pcap': 'D1 (1 jam)',        'n_flow': 3687, 'n_false_alarm': 14, 'far': 0.003797},
    ]

dff = pd.DataFrame(far_rows)
cols = [c for c in ['pcap','n_flow','n_false_alarm','far'] if c in dff.columns]
dff = dff[cols].copy()
dff['FAR (%)'] = (dff['far']*100).round(4)
display(dff)

In [ ]:
# Tabel perbandingan FAR antar-durasi (isi otomatis dari D1..D5 bila tersedia)
ramp = pd.DataFrame([
    {'Tahap': 'S0 (3 menit)', 'n_flow': 203,   'false_alarm': 0,  'FAR (%)': 0.0000, 'Status': 'gate LOLOS'},
    {'Tahap': 'D1 (1 jam)',   'n_flow': 3687,  'false_alarm': 14, 'FAR (%)': 0.3797, 'Status': 'gate LOLOS'},
    {'Tahap': 'D2 (6 jam)',   'n_flow': 22644, 'false_alarm': 94, 'FAR (%)': 0.4152, 'Status': 'gate LOLOS (0,36-0,48%/jam)'},
    {'Tahap': 'D3 (24 jam)',  'n_flow': None,  'false_alarm': None, 'FAR (%)': None,  'Status': 'dilewati sementara'},
])
display(ramp)

sub = ramp.dropna(subset=['FAR (%)'])
fig, ax = plt.subplots(figsize=(6.0, 3.4))
bars = ax.bar(sub['Tahap'], sub['FAR (%)'], color='#4C72B0')
ax.set_ylabel('FAR (%)'); ax.set_title('FAR trafik nyata (benign) per durasi observasi')
for b, v in zip(bars, sub['FAR (%)']):
    ax.text(b.get_x()+b.get_width()/2, v+0.01, f'{v:.4f}%', ha='center', fontweight='bold')
ax.set_ylim(0, max(0.6, sub['FAR (%)'].max()*1.4))
plt.tight_layout(); plt.show()
print('FAR rendah & konsisten -> detektor tidak cerewet pada trafik normal nyata.')

## 7b. Deteksi Serangan di AWS (Fase 2, BINER) — Temuan Distribution Shift

Fase 2 menyalakan Attacker + Target: serangan nyata dilancarkan, direkam, lalu dinilai model
UNSW (Model A 9-fitur) tanpa pelatihan ulang. Dua varian diuji:
- **clean**: SSH brute-force + Slowloris + SYN flood (200/s) — serangan 'lambat'.
- **volumetric**: SYN flood 5000/s + HTTP flood cepat + UDP flood — serangan laju-tinggi
  (dirancang agar profil `src_load`/`dst_load` mendekati kelas DoS/Generic UNSW).

**Hasil: model TIDAK mendeteksi serangan AWS (recall ~0) pada kedua varian.** Ini BUKAN
feature-mismatch/satuan (z-score fitur wajar, SFM & unit benar), melainkan **distribution shift**
pada distribusi fitur *laju* (`src_load`, `dst_load`): flow AWS berdurasi panjang (~64 detik)
sehingga laju = byte/durasi menjadi RENDAH, sedangkan serangan UNSW berupa flow PENDEK ber-laju
TINGGI. Memperbesar volume serangan (volumetric) TIDAK menutup celah ini. Temuan ini konsisten
dengan tesis: keunggulan benchmark tak otomatis mentransfer ke trafik nyata; solusinya kalibrasi
domain (few-shot/mixup), bukan mengubah serangan.

### Definisi metrik & rumus (untuk membaca tabel/gambar di bawah)

**Confusion matrix biner** (positif = *attack*, negatif = *normal*). Empat komponen penyusun:

| Simbol | Nama | Arti |
|---|---|---|
| **TP** | True Positive | flow **attack** diprediksi **attack** (benar) |
| **TN** | True Negative | flow **normal** diprediksi **normal** (benar) |
| **FP** | False Positive | flow **normal** diprediksi **attack** (alarm palsu) |
| **FN** | False Negative | flow **attack** diprediksi **normal** (serangan **lolos**) |

**Rumus:**

$$\text{Recall} = \frac{TP}{TP + FN}$$

> Dari semua flow yang **benar-benar serangan**, berapa yang **berhasil ditangkap**. Recall rendah
> = banyak serangan **lolos** (FN besar). Inilah metrik utama di sini: recall $\approx 0$ berarti
> model melewatkan hampir semua serangan AWS.

$$\text{Precision} = \frac{TP}{TP + FP}$$

> Dari semua yang **diprediksi serangan**, berapa yang **benar** serangan. Di data AWS ini
> Precision $=1{,}0$ **menyesatkan**: model sangat jarang menebak 'attack', dan yang sedikit itu
> kebetulan benar (FP$=0$) — jadi presisi sempurna tapi tak berguna karena recall $\approx 0$.

$$\text{F1} = \frac{2 \cdot \text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

> Rata-rata harmonik Precision & Recall. Ikut rendah karena recall rendah.

$$\text{MCC} = \frac{TP\cdot TN - FP\cdot FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}}$$

> *Matthews Correlation Coefficient*: memakai **keempat** komponen (TP, TN, FP, FN) sekaligus,
> sehingga **tahan terhadap ketidakseimbangan kelas**. Rentang $-1$ (selalu salah) .. $0$ (setara
> tebakan acak) .. $+1$ (sempurna). MCC $\approx 0$ di sini = deteksi setara **menebak acak**.

**Catatan penyajian:**
- Gambar bawah menampilkan **Recall** karena data Fase 2 hampir seluruhnya serangan (mis. volumetric
  17.287 attack / 17.319 flow) → akurasi/precision menyesatkan; **recall paling jujur** menunjukkan
  kegagalan deteksi.
- Hanya **model UNSW** yang tampil karena **hanya model itu yang di-deploy & diuji di AWS**
  (satu model biner 9-fitur). Model CIC **tidak diuji di AWS**, jadi datanya memang tak ada
  (bukan disembunyikan).

In [ ]:
# Hasil deteksi Fase 2 di AWS (angka nyata dari detect_*_metrics.json di S3).
det = pd.DataFrame([
    {'Varian': 'clean (brute+slowloris+SYN 200/s)', 'n_flow': 434,   'attack_gt': 333,   'MCC': 0.0265, 'F1': 0.006,  'Precision': 1.0, 'Recall': 0.003},
    {'Varian': 'volumetric (SYN 5000/s+HTTP+UDP)', 'n_flow': 17319, 'attack_gt': 17287, 'MCC': 0.0007, 'F1': 0.0006, 'Precision': 1.0, 'Recall': 0.0003},
])
print('Tabel deteksi AWS (Fase 2) - model UNSW tanpa kalibrasi:')
display(det)

# Diagnosa: profil fitur laju attack AWS vs mean training (nilai nyata)
diag = pd.DataFrame([
    {'Fitur': 'duration (us)', 'attack AWS (median)': 64031000.0, 'train mean': 12175143.81},
    {'Fitur': 'src_load',      'attack AWS (median)': 15.58,       'train mean': 255117.10},
    {'Fitur': 'dst_load',      'attack AWS (median)': 0.12,        'train mean': 15260.12},
]).round(2)
print('\nDiagnosa distribution shift (volumetric) - fitur laju jauh di bawah training:')
display(diag)

fig, ax = plt.subplots(figsize=(6.0, 3.4))
labels = ['clean', 'volumetric']
recalls = [0.003, 0.0003]
bars = ax.bar(labels, recalls, color='#C44E52')
ax.set_ylabel('Recall (attack)'); ax.set_ylim(0, 0.01)
ax.set_title('Recall model UNSW terhadap serangan AWS ~ 0 (tanpa kalibrasi)')
for b, v in zip(bars, recalls):
    ax.text(b.get_x()+b.get_width()/2, v+0.0002, f'{v:.4f}', ha='center', fontweight='bold')
plt.tight_layout(); plt.show()
print('Akar: src_load/dst_load rendah krn durasi flow panjang -> distribution shift, bukan feature mismatch.')
print('Langkah lanjut yang direncanakan: kalibrasi few-shot dgn sampel domain AWS (data CSV sudah di S3).')

## 7c. Distribution Shift 3-Domain (CIC / UNS / AWS) — Kuantitatif

Verifikasi bahwa kegagalan zero-shot berakar pada *distribution shift*, bukan artefak alat ukur.
Membandingkan distribusi 9 fitur SFM antar tiga domain memakai **jarak Wasserstein** (a la Layeghy)
dan **domain-classifier** (akurasi → 1 = domain terpisah sempurna). Satuan laju UNS sudah dikoreksi
(bit→byte /8; `dst_load`=dpkts/dur) agar setara CIC/AWS. Angka dari `dataset_shift_results.json`
(hasil notebook `21_dataset_shift.ipynb`, ter-upload ke S3 `unsw-far/shift/`).

In [ ]:
# Shift 3-domain: muat dari JSON hasil notebook 21 bila ada; jika tidak, pakai angka nyata (setelah koreksi satuan).
shift = load_json('dataset_shift_results.json', fallback={
    'n_flow': {'CIC': 1623261, 'UNS': 172684, 'AWS-real': 44287},
    'wasserstein': [
        {'pasangan': 'CIC vs UNS',      'RATA2': 0.317},
        {'pasangan': 'CIC vs AWS-real', 'RATA2': 0.230},
        {'pasangan': 'UNS vs AWS-real', 'RATA2': 0.372},
    ],
    'domain_classifier': [
        {'pasangan': 'CIC vs UNS',      'akurasi_pembeda': 0.990, 'proxy_A_distance': 1.961},
        {'pasangan': 'CIC vs AWS-real', 'akurasi_pembeda': 0.994, 'proxy_A_distance': 1.975},
        {'pasangan': 'UNS vs AWS-real', 'akurasi_pembeda': 0.998, 'proxy_A_distance': 1.993},
    ],
})
wass = {r['pasangan']: r['RATA2'] for r in shift['wasserstein']}
dc = {r['pasangan']: r for r in shift['domain_classifier']}
tab_shift = pd.DataFrame([
    {'Pasangan': p, 'W1 rata-rata': round(wass[p], 3),
     'Akurasi pembeda': dc[p]['akurasi_pembeda'], 'Proxy A-distance': dc[p]['proxy_A_distance']}
    for p in wass
])
print('Jumlah flow per domain:', shift['n_flow'])
print('\nShift 3-domain (Wasserstein + domain-classifier):')
display(tab_shift)
print('Interpretasi: akurasi pembeda ~0,99 di semua pasangan -> ketiga domain terpisah (distribution shift nyata).')
print('=== SEL 8c (shift 3-domain) SELESAI ===')

## 7d. Repair via Few-Shot ke AWS — MCC Pulih dari ≈ 0

Model source (UNS/CIC) di-*repair* dengan sedikit label AWS. Uji pada AWS-test tetap (7.102 flow;
7.049 attack, 53 benign). Angka dari `aws_fewshot_results.json` (notebook `22_aws_fewshot.ipynb`,
S3 `unsw-far/fewshot/`). **Catatan:** karena test 99% attack, F1 tinggi menyesatkan — **MCC** adalah
metrik utama di sini.

In [ ]:
# Few-shot AWS: muat JSON hasil notebook 22 bila ada; jika tidak, pakai angka nyata dua-source.
fs = load_json('aws_fewshot_results.json', fallback={'runs': [
    {'source':'UNS','k_percent':0.0,'mcc':0.0736},{'source':'UNS','k_percent':1.0,'mcc':0.4899},
    {'source':'UNS','k_percent':5.0,'mcc':0.4787},{'source':'UNS','k_percent':10.0,'mcc':0.5879},
    {'source':'UNS','k_percent':20.0,'mcc':0.5955},{'source':'UNS','k_percent':50.0,'mcc':0.5739},
    {'source':'CIC','k_percent':0.0,'mcc':-0.2172},{'source':'CIC','k_percent':1.0,'mcc':0.1315},
    {'source':'CIC','k_percent':5.0,'mcc':0.3696},{'source':'CIC','k_percent':10.0,'mcc':0.2073},
    {'source':'CIC','k_percent':20.0,'mcc':0.4562},{'source':'CIC','k_percent':50.0,'mcc':0.1349},
    {'source':'AWS','k_percent':100.0,'mcc':0.5975,'mode':'AWS-only (acuan atas)'},
]})
runs = fs['runs']
ks = [0,1,5,10,20,50]
def mcc_of(src,k): return next((r['mcc'] for r in runs if r.get('source')==src and r['k_percent']==k), None)
tab_fs = pd.DataFrame({'% label AWS': ks,
                       'UNS+AWS (MCC)': [mcc_of('UNS',k) for k in ks],
                       'CIC+AWS (MCC)': [mcc_of('CIC',k) for k in ks]})
aws_only = next((r['mcc'] for r in runs if str(r.get('mode','')).startswith('AWS-only') or r['k_percent']==100.0), None)
print('Few-shot ke AWS (MCC di AWS-test). k=0 = zero-shot; AWS-only (acuan atas) =', aws_only)
display(tab_fs)

fig, ax = plt.subplots(figsize=(7.0, 4.0))
for src, col in [('UNS','#DD8452'), ('CIC','#4C72B0')]:
    ys = [mcc_of(src,k) for k in ks]
    if any(v is not None for v in ys):
        ax.plot(ks, ys, 'o-', lw=2, color=col, label=f'{src}+AWS')
if aws_only is not None:
    ax.axhline(aws_only, ls='--', color='#55A868', label=f'AWS-only={aws_only:.3f}')
ax.set_xlabel('% label AWS pada train'); ax.set_ylabel('MCC di AWS-test'); ax.set_ylim(-0.3, 1.0)
ax.set_title('Few-shot domain calibration ke trafik nyata AWS'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print('Kesimpulan: 1% label AWS sudah melompatkan MCC dari ~0/negatif; 10-20% menyamai AWS-only.')
print('=== SEL 8d (few-shot AWS) SELESAI ===')

## 7e. Matriks Pergeseran Fitur Per-Kategori (Biner vs Multi-Kelas)

Menjawab pertanyaan reviewer: evaluasi kita biner (attack vs benign), tetapi NIDS operasional
sering menuntut multi-kelas. Di sini kita tunjukkan **mengapa** adaptasi few-shot multi-kelas lebih
sulit, secara kuantitatif. Untuk tiap dataset kita pulihkan taksonomi serangan aslinya (label CIC
tersimpan di CSV mentah `file_100.csv`; UNSW punya `attack_cat`), kelompokkan ke kategori kasar
yang sepadan, standardisasi 9 fitur SFM per-dataset, lalu ukur jarak Wasserstein $W_1$ tiap kategori
serangan ke benign-nya sendiri. Angka dari `category_shift_results.json` / `category_shift_matrix.csv`
(notebook `23_category_shift.ipynb`, S3 `unsw-far/catshift/`).

**Temuan baru:** shift bersifat **bergantung-kategori** (bukan seragam), dan kategori yang nominalnya
sama bergeser **berbeda antar-jaringan** -> satu kalibrasi global tunggal tidak cukup untuk multi-kelas.

In [ ]:
# Matriks shift per-kategori (muat CSV/JSON hasil notebook 23 bila ada; jika tidak, pakai angka nyata terekam).
CANON9 = ['duration','fwd_pkts','bwd_pkts','fwd_bytes','bwd_bytes','fwd_mean','bwd_mean','src_load','dst_load']
_cat_csv = next((os.path.join(d,'category_shift_matrix.csv') for d in [DATA_DIR,'.','catshift_out']
                 if os.path.exists(os.path.join(d,'category_shift_matrix.csv'))), None)
if _cat_csv:
    catm = pd.read_csv(_cat_csv); print('loaded:', _cat_csv)
else:
    print('(fallback) memakai angka nyata terekam dari catshift.')
    catm = pd.DataFrame([
        {'dataset':'CIC','category':'DDoS','n':126389,'duration':0.0219,'fwd_pkts':0.1556,'bwd_pkts':0.0361,'fwd_bytes':0.1573,'bwd_bytes':0.0232,'fwd_mean':0.7024,'bwd_mean':0.4186,'src_load':0.0748,'dst_load':0.0471,'MEAN':0.1819},
        {'dataset':'CIC','category':'DoS','n':65426,'duration':0.0292,'fwd_pkts':0.0027,'bwd_pkts':0.0413,'fwd_bytes':0.0077,'bwd_bytes':0.0238,'fwd_mean':0.7783,'bwd_mean':0.6533,'src_load':0.0826,'dst_load':0.8064,'MEAN':0.2695},
        {'dataset':'CIC','category':'BruteForce','n':38174,'duration':0.034,'fwd_pkts':0.0031,'bwd_pkts':0.0417,'fwd_bytes':0.0067,'bwd_bytes':0.0224,'fwd_mean':0.5871,'bwd_mean':0.5763,'src_load':0.0833,'dst_load':3.6314,'MEAN':0.554},
        {'dataset':'CIC','category':'Botnet','n':28618,'duration':0.0342,'fwd_pkts':0.0027,'bwd_pkts':0.036,'fwd_bytes':0.0069,'bwd_bytes':0.0239,'fwd_mean':0.5345,'bwd_mean':0.657,'src_load':0.0815,'dst_load':0.0474,'MEAN':0.1582},
        {'dataset':'CIC','category':'Infiltration','n':16047,'duration':0.0079,'fwd_pkts':0.0009,'bwd_pkts':0.0167,'fwd_bytes':0.0059,'bwd_bytes':0.0143,'fwd_mean':0.2995,'bwd_mean':0.1448,'src_load':0.0229,'dst_load':0.3924,'MEAN':0.1006},
        {'dataset':'UNSW','category':'Generic','n':18860,'duration':0.2063,'fwd_pkts':0.154,'bwd_pkts':0.2131,'fwd_bytes':0.0231,'bwd_bytes':0.1174,'fwd_mean':0.4522,'bwd_mean':0.6738,'src_load':0.4391,'dst_load':0.1747,'MEAN':0.2726},
        {'dataset':'UNSW','category':'Exploits','n':11125,'duration':0.1434,'fwd_pkts':0.1381,'bwd_pkts':0.0578,'fwd_bytes':0.172,'bwd_bytes':0.0563,'fwd_mean':0.2284,'bwd_mean':0.2043,'src_load':0.1634,'dst_load':0.1948,'MEAN':0.1509},
        {'dataset':'UNSW','category':'Fuzzers','n':6048,'duration':0.2302,'fwd_pkts':0.0877,'bwd_pkts':0.1743,'fwd_bytes':0.0111,'bwd_bytes':0.1225,'fwd_mean':0.2732,'bwd_mean':0.5754,'src_load':0.3761,'dst_load':0.1729,'MEAN':0.2248},
        {'dataset':'UNSW','category':'DoS','n':4086,'duration':0.2938,'fwd_pkts':0.1882,'bwd_pkts':0.27,'fwd_bytes':0.1137,'bwd_bytes':0.2238,'fwd_mean':0.2283,'bwd_mean':0.475,'src_load':0.4556,'dst_load':0.173,'MEAN':0.269},
        {'dataset':'UNSW','category':'Recon','n':3494,'duration':0.0564,'fwd_pkts':0.1214,'bwd_pkts':0.1888,'fwd_bytes':0.0203,'bwd_bytes':0.1238,'fwd_mean':0.4095,'bwd_mean':0.6196,'src_load':0.2632,'dst_load':0.1732,'MEAN':0.2196},
    ])

show_cols = ['dataset','category','n','MEAN'] + CANON9
print('Matriks W1 per-kategori ke benign (per-dataset, z-space):')
display(catm[[c for c in show_cols if c in catm.columns]].round(3))

# Fitur paling bergeser per baris
print('\nFitur pendorong shift (terbesar) per kategori:')
for _, r in catm.iterrows():
    vals = {f: r[f] for f in CANON9 if f in catm.columns}
    top = max(vals, key=vals.get)
    print(f"  {r['dataset']:4s} {r['category']:14s} -> {top} (W1={vals[top]:.3f})")
print('=== SEL 8e-1 (tabel per-kategori) SELESAI ===')

In [ ]:
# Heatmap kategori x fitur (dipisah dataset lewat label baris).
rows_lbl = [f"{r.dataset}:{r.category}" for r in catm.itertuples()]
M = catm[[c for c in CANON9 if c in catm.columns]].values
fig, ax = plt.subplots(figsize=(9, 0.8 + 0.5*len(catm)))
im = ax.imshow(M, aspect='auto', cmap='YlOrRd')
ax.set_xticks(range(len(CANON9))); ax.set_xticklabels(CANON9, rotation=45, ha='right')
ax.set_yticks(range(len(rows_lbl))); ax.set_yticklabels(rows_lbl)
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        ax.text(j, i, f'{M[i,j]:.2f}', ha='center', va='center', fontsize=7)
fig.colorbar(im, ax=ax, label='W1 ke benign')
ax.set_title('Matriks pergeseran fitur per-kategori, lintas-dataset (W1 ke benign)')
plt.tight_layout(); plt.show()

# Bandingkan kategori sepadan lintas-dataset: DoS CIC vs DoS UNSW.
try:
    dos_cic = catm[(catm.dataset=='CIC') & (catm.category=='DoS')].iloc[0]
    dos_uns = catm[(catm.dataset=='UNSW') & (catm.category=='DoS')].iloc[0]
    print('DoS lintas-dataset -> rata2 shift hampir sama, tapi fitur dominan BEDA:')
    print(f"  CIC  DoS: MEAN={dos_cic['MEAN']:.3f}, dominan={max(CANON9, key=lambda f: dos_cic[f])}")
    print(f"  UNSW DoS: MEAN={dos_uns['MEAN']:.3f}, dominan={max(CANON9, key=lambda f: dos_uns[f])}")
    print('  -> kalibrasi DoS satu jaringan belum tentu transfer ke jaringan lain.')
except Exception as e:
    print('(lewati perbandingan DoS:', e, ')')
print('=== SEL 8e-2 (heatmap per-kategori) SELESAI ===')

## 7f. Analisis Ketahanan: Split Episode/Temporal + Selang Kepercayaan (CI)

Menjawab pertanyaan reviewer soal **temporal/episode leakage** pada few-shot AWS. Angka §7d
memakai *split* tingkat-*flow* berlapis (test 99% attack). Untuk menguji apakah pemulihan
few-shot itu **adaptasi lintas-episode sesungguhnya** atau sekadar efek komposisi test, kita
tambah dua *split* lebih ketat + **bootstrap 95% CI** untuk MCC (notebook `26`/`27`,
S3 `unsw-far/temporal/` & `unsw-far/episode/`).

- **Split episode:** latih (kalibrasi) pada episode {BruteForce, DoS} + separuh benign, uji pada
  episode **DDoS yang tak pernah dilihat** + separuh benign (204 flow: 121 DDoS, 83 benign).
- **Split temporal:** potong waktu 60/40. Kurang informatif di sesi ini (serangan terkonsentrasi
  menit 1–5, sisanya benign) — dilaporkan sebagai keterbatasan desain.

**Temuan jujur:** di split episode, model sumber-UNS **sudah zero-shot kuat** untuk DDoS
(MCC 0,58, melampaui AWS-only 0,42), dan few-shot **tidak** menaikkannya. Artinya pemulihan
few-shot AWS yang tajam di §7d **sebagian bergantung komposisi test**. Bukti terkuat adaptasi
few-shot tetap pada eksperimen **offline** lintas-jaringan (§5). Ini melemahkan sedikit klaim
AWS few-shot tetapi **memperkuat kredibilitas** — dilaporkan apa adanya.

In [ ]:
# Robustness AWS: split episode (train BruteForce+DoS, test DDoS tak-terlihat) + bootstrap 95% CI.
# Angka nyata dari notebook 27 (aws_episode_results.json). Fallback = hasil terekam.
epi = load_json('aws_episode_results.json', fallback=None)
_epi_paths = ['episode_split_out/aws_episode_results.json', os.path.join(DATA_DIR,'aws_episode_results.json')]
if epi is None:
    for _p in _epi_paths:
        if os.path.exists(_p): epi = json.load(open(_p)); print('loaded:', _p); break

if epi and epi.get('runs'):
    rows = []
    for r in epi['runs']:
        rows.append({'Sumber': r['source'], 'Mode': r['mode'], 'k%': r['k_percent'],
                     'MCC': r['mcc'], 'CI_lo': r.get('mcc_ci_lo'), 'CI_hi': r.get('mcc_ci_hi'),
                     'Recall': r['recall'], 'Precision': r['precision']})
    dfe = pd.DataFrame(rows)
    print('Split EPISODE (uji DDoS tak-terlihat) — MCC + 95% CI:')
    display(dfe.round(3))
else:
    print('(fallback) angka nyata terekam dari notebook 27:')
    dfe = pd.DataFrame([
        {'Sumber':'UNS','Mode':'zero-shot','k%':0,  'MCC':0.579,'CI_lo':0.46,'CI_hi':0.70},
        {'Sumber':'UNS','Mode':'few-shot 5%','k%':5,'MCC':0.523,'CI_lo':0.40,'CI_hi':0.64},
        {'Sumber':'UNS','Mode':'few-shot 20%','k%':20,'MCC':0.420,'CI_lo':0.29,'CI_hi':0.54},
        {'Sumber':'CIC','Mode':'zero-shot','k%':0, 'MCC':-0.220,'CI_lo':-0.34,'CI_hi':-0.08},
        {'Sumber':'AWS','Mode':'AWS-only','k%':100,'MCC':0.418,'CI_lo':0.29,'CI_hi':0.54},
    ])
    display(dfe.round(3))

# Bar chart: zero-shot vs few-shot(5%) per source, dengan error bar CI (source UNS & CIC)
try:
    fig, ax = plt.subplots(figsize=(7, 4))
    plot_rows = dfe[dfe['Sumber'].isin(['UNS','CIC'])]
    labels = [f"{r.Sumber}\n{r.Mode}" for r in plot_rows.itertuples()]
    mccs = plot_rows['MCC'].values
    lo = plot_rows['MCC'].values - plot_rows['CI_lo'].values
    hi = plot_rows['CI_hi'].values - plot_rows['MCC'].values
    colors = ['#4C72B0' if s=='UNS' else '#C44E52' for s in plot_rows['Sumber']]
    ax.bar(range(len(mccs)), mccs, yerr=[lo, hi], capsize=4, color=colors, edgecolor='k')
    ax.axhline(0, color='k', lw=0.8)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, fontsize=8)
    ax.set_ylabel('MCC (episode split, uji DDoS)')
    ax.set_title('Robustness AWS: zero-shot UNS sudah kuat; few-shot tak menambah (95% CI)')
    plt.tight_layout(); plt.show()
except Exception as e:
    print('(lewati bar chart:', e, ')')
print('=== SEL 7f (robustness episode/temporal + CI) SELESAI ===')

# ===================================================================
# BAGIAN MULTI-CLASS (setelah seluruh alur biner selesai)
# ===================================================================

## 8. Klasifikasi MULTI-CLASS — Pengujian OFFLINE (kategori serangan)

> **Alur presentasi:** Bagian 3–7 menyelesaikan cerita **BINER** (training → testing offline →
> SFM → diagnosis/few-shot → testing AWS → efisiensi). Mulai **Bagian 8** kita ulangi alur serupa
> untuk **MULTI-CLASS** (prediksi *kategori* serangan, bukan sekadar attack/benign).

**Cara & skenario pengujian (multi-class, offline):**
- **Label:** taksonomi asli tiap dataset dipetakan ke kategori kasar — CIC: `Benign, DoS, DDoS,
  BruteForce, Botnet, Infiltration, Web`; UNSW: `Benign, Generic, Exploits, Fuzzers, DoS, Recon,
  ...`. Kelas dgn sampel sangat sedikit (<200) digabung `Other-rare` agar stabil.
- **Model:** XGBoost `multi:softprob` (`num_class`=jumlah kategori), config sama seperti biner.
- **Skema uji:** **in-domain per dataset** (split 70/30, scaler fit-on-train). Metrik: **macro-F1**
  (rata antar-kelas, adil utk kelas minor), **MCC multi-kelas**, **balanced-accuracy**, dan
  **confusion matrix** (recall per-kategori).
- **Cross-dataset multi-class** hanya pada kategori sepadan (mis. Benign, DoS) karena taksonomi
  kedua dataset berbeda.

> Notebook sumber: `24_multiclass.ipynb` -> `multiclass_results.json` (S3 `unsw-far/multiclass/`).

In [ ]:
# Multi-class OFFLINE (in-domain per dataset) dari notebook 24. Angka nyata bila JSON tersedia.
mc_off = load_json('multiclass_results.json', fallback=None)
_mc_paths = ['multiclass_out/multiclass_results.json', os.path.join(DATA_DIR, 'multiclass_results.json')]
if mc_off is None:
    for _p in _mc_paths:
        if os.path.exists(_p): mc_off = json.load(open(_p)); print('loaded', _p); break

if mc_off and mc_off.get('in_domain'):
    rows = []
    for nm, r in mc_off['in_domain'].items():
        rows.append({'dataset': nm, '#kelas': len(r['classes']), 'macro-F1': r['macro_f1'],
                     'weighted-F1': r['weighted_f1'], 'MCC': r['mcc'], 'balanced-acc': r['balanced_acc']})
    display(pd.DataFrame(rows))
    for nm, r in mc_off['in_domain'].items():
        labels = r['classes']; cm = np.array(r['confusion'], float)
        row = cm.sum(1, keepdims=True); row[row == 0] = 1; cmn = cm / row
        fig, ax = plt.subplots(figsize=(1.3 + 0.7*len(labels), 1.1 + 0.7*len(labels)))
        im = ax.imshow(cmn, cmap='Blues', vmin=0, vmax=1)
        ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=45, ha='right')
        ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
        ax.set_xlabel('Prediksi'); ax.set_ylabel('Sebenarnya')
        for i in range(cmn.shape[0]):
            for j in range(cmn.shape[1]):
                ax.text(j, i, f'{cmn[i,j]:.2f}', ha='center', va='center', fontsize=7,
                        color='white' if cmn[i,j] > 0.5 else 'black')
        ax.set_title(f'{nm} MULTI-CLASS offline (recall ternormalisasi)')
        fig.colorbar(im, ax=ax, fraction=0.046); plt.tight_layout(); plt.show()
        display(pd.DataFrame(r['per_class']).T[['precision', 'recall', 'f1', 'support']])
    print('Multi-class in-domain: kategori bervolume tinggi (DDoS/DoS, Generic) terklasifikasi baik;')
    print('kategori langka lebih sulit -> konsisten dgn class-imbalance.')
else:
    print('multiclass_results.json belum tersedia. Jalankan 24_multiclass.ipynb di SageMaker,')
    print('unduh hasil dari s3://.../unsw-far/multiclass/ ke folder unswnb-15/, lalu jalankan sel ini lagi.')

## 8b. Klasifikasi MULTI-CLASS — Pengujian di AWS (Fase 2)

Melanjutkan alur multi-class ke **trafik nyata AWS**. Karena serangan dijalankan **terjadwal
satu jenis per selang waktu**, label **kategori** (Benign/BruteForce/DoS/DDoS) diturunkan dari
**timeline** — bukan hanya attack/benign.

**Cara & skenario pengujian (multi-class, AWS):**
- **Timeline ground-truth** (contoh): 0–1 mnt Benign; 1–5 BruteForce (SSH+FTP); 5–9 DoS
  (Slowloris + HTTP flood); 9–10 DDoS (SYN flood); 10–12 Benign.
- **Model:** XGBoost `multi:softprob` dilatih pada kategori **CIC** (kelas beririsan dgn AWS:
  Benign, BruteForce, DoS, DDoS), diuji **zero-shot** ke AWS + **few-shot** (tambah k% label AWS).
- **Metrik:** macro-F1, balanced-accuracy, confusion matrix (recall per-kategori).

> Notebook sumber: `25_multiclass_aws.ipynb` -> `multiclass_aws_results.json`
> (S3 `unsw-far/multiclass_aws/`). **Butuh sumber berlabel-kategori** (pcap AWS + NFStream, atau
> CSV dgn `elapsed_sec`/`ground_truth`). Bila hanya ada CSV biner, multi-class AWS tak bisa dihitung.

In [ ]:
# Multi-class AWS (Fase 2) dari notebook 25. Angka nyata bila JSON tersedia; jika tidak, beri catatan jujur.
mca = load_json('multiclass_aws_results.json', fallback=None)
_mca_paths = ['multiclass_aws_out/multiclass_aws_results.json', os.path.join(DATA_DIR, 'multiclass_aws_results.json')]
if mca is None:
    for _p in _mca_paths:
        if os.path.exists(_p): mca = json.load(open(_p)); print('loaded', _p); break

def _plot_cm_aws(cm, labels, title):
    M = np.array(cm, float); row = M.sum(1, keepdims=True); row[row == 0] = 1; M = M / row
    fig, ax = plt.subplots(figsize=(1.3 + 0.8*len(labels), 1.1 + 0.8*len(labels)))
    im = ax.imshow(M, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
    ax.set_xlabel('Prediksi'); ax.set_ylabel('Sebenarnya')
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax.text(j, i, f'{M[i,j]:.2f}', ha='center', va='center', fontsize=8, color='white' if M[i,j] > 0.5 else 'black')
    ax.set_title(title); fig.colorbar(im, ax=ax, fraction=0.046); plt.tight_layout(); plt.show()

if mca and mca.get('zero_shot'):
    zs = mca['zero_shot']; labels = zs['classes']
    print(f"AWS multi-class ZERO-SHOT (CIC->AWS): macro-F1={zs['macro_f1']} bal-acc={zs['balanced_acc']} (n={zs['n']})")
    display(pd.DataFrame(zs['per_class']).T)
    _plot_cm_aws(zs['confusion'], labels, 'AWS multi-class zero-shot (recall)')
    if mca.get('few_shot'):
        fsdf = pd.DataFrame(mca['few_shot'])
        display(fsdf)
        fig, ax = plt.subplots(figsize=(6.2, 3.6))
        ax.plot(fsdf['k_percent'], fsdf['macro_f1'], 'o-', color='#4C72B0', label='macro-F1')
        ax.plot(fsdf['k_percent'], fsdf['balanced_acc'], 's-', color='#DD8452', label='balanced-acc')
        ax.set_xlabel('% label AWS pada train'); ax.set_ylabel('skor'); ax.set_ylim(0, 1)
        ax.set_title('AWS multi-class few-shot (tambah k% label AWS)'); ax.legend(); plt.tight_layout(); plt.show()
    print('Interpretasi: zero-shot multi-class AWS umumnya rendah (distribution shift, sama seperti biner);')
    print('few-shot menambah sedikit label AWS memperbaiki kategori -> konsisten dgn temuan biner.')
else:
    print('multiclass_aws_results.json belum tersedia / hanya biner (zero_shot=null).')
    print('Jalankan 25_multiclass_aws.ipynb di SageMaker dgn sumber berlabel-kategori')
    print('(pcap AWS + NFStream, atau CSV dgn elapsed_sec/ground_truth), lalu unduh dari')
    print('s3://.../unsw-far/multiclass_aws/ ke folder unswnb-15/ dan jalankan sel ini lagi.')

## 9. Diagram Alur Pipeline (Paket -> Keputusan)

In [ ]:
# Diagram alur sederhana pakai matplotlib (tanpa dependensi tambahan)
fig, ax = plt.subplots(figsize=(11, 2.2))
ax.axis('off')
steps = ['Paket jaringan\n(ens5)', 'tcpdump\n-> .pcap', 'NFStream\nflow + statistik',
         '9 fitur SFM\n(+konversi satuan)', 'z-score\n(scaler latih)', 'XGBoost\npredict',
         'Label:\nnormal / attack']
n = len(steps); x = np.linspace(0.02, 0.98, n)
for i, (xi, s) in enumerate(zip(x, steps)):
    ax.add_patch(plt.Rectangle((xi-0.06, 0.35), 0.12, 0.3, fc='#EAF0F7', ec='#4C72B0', lw=1.5))
    ax.text(xi, 0.5, s, ha='center', va='center', fontsize=9)
    if i < n-1:
        ax.annotate('', xy=(x[i+1]-0.065, 0.5), xytext=(xi+0.065, 0.5),
                    arrowprops=dict(arrowstyle='->', color='#333', lw=1.4))
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title('Alur validasi trafik nyata: dari paket ke keputusan', fontsize=11)
plt.tight_layout(); plt.show()

## 9b. Galeri Gambar Siap-Pakai (dari `figure-q1/`)

Gambar-gambar publikasi yang sudah dibuat sebelumnya (via `make_figures.py`). Bila folder
`figure-q1/` tersedia, sel di bawah menampilkannya langsung sehingga mudah disalin ke slide.

In [ ]:
from matplotlib import image as mpimg

FIG_CANDIDATES = [os.path.join(DATA_DIR, 'figure-q1'), 'figure-q1', '../figure-q1']
FIG_DIR = next((d for d in FIG_CANDIDATES if os.path.isdir(d)), None)
print('FIG_DIR =', os.path.abspath(FIG_DIR) if FIG_DIR else 'tidak ditemukan')

figs = [
    ('fig1_generalization_gap.png', 'Celah generalisasi in-domain vs lintas-jaringan'),
    ('fig2_fewshot_curve.png',      'Kurva few-shot: 1% label target memulihkan MCC'),
    ('fig3_alignment.png',          'Strategi penyelarasan (baseline / CORAL / joint)'),
    ('fig4_functional_evasion.png', 'Functional-preserving evasion vs FGSM tak-terbatas'),
    ('fig5_adaptive_whitebox.png',  'Evaluasi ketahanan adversarial (referensi paper)'),
    ('fig6_deployment_blueprint.png','Blueprint penyebaran (edge / real-traffic)'),
    ('fig_shift_wasserstein.png',   'Distribution shift 3-domain (Wasserstein)'),
    ('aws_fewshot_curve.png',       'Few-shot ke AWS: MCC pulih dari ~0'),
]

if FIG_DIR:
    for fname, cap in figs:
        p = os.path.join(FIG_DIR, fname)
        if os.path.exists(p):
            img = mpimg.imread(p)
            fig, ax = plt.subplots(figsize=(7.5, 7.5*img.shape[0]/img.shape[1]))
            ax.imshow(img); ax.axis('off'); ax.set_title(cap, fontsize=10)
            plt.tight_layout(); plt.show()
        else:
            print('  (lewati, tak ada):', fname)
else:
    print('Folder figure-q1/ tidak ditemukan; lewati galeri. Diagram di sel sebelumnya tetap tersedia.')

## 10. Rangkuman Temuan Utama (untuk slide penutup)

1. **SFM valid**: 9 fitur irisan cukup ekspresif (joint training ~ in-domain di kedua jaringan).
2. **NIDS single-source runtuh lintas-jaringan** (MCC ~0): fitur terbukti *cukup* (lihat #1),
   penyebab *dominan* adalah *distribution shift* yang terukur langsung (Wasserstein + domain-classifier ~0.99).
   *(Klaim dibatasi: faktor taksonomi-label & pilihan classifier diakui belum diisolasi penuh -- lihat Bagian 11.)*
3. **Kalibrasi minimal 1% label target** (atau *mixup* tanpa label) memulihkan MCC ke 0.65-0.91.
4. **Model ringan** (~2.9 MB, ratusan us/flow, ribuan flow/detik di 1 vCPU) -> layak *edge*.
   *(Catatan: evaluasi adversarial TIDAK dibahas di rangkuman ini; ada tema terpisah pada notebook `13_rangkuman_adversarial.ipynb`.)*
5. **Validasi trafik nyata AWS (Fase 1 FAR)**: FAR rendah & stabil pada trafik benign
   (S0 0%, D1 0.38%, D2 0.42% lintas 6 jam) -> detektor tidak cerewet di dunia nyata.
6. **Deteksi di AWS (Fase 2)**: model UNSW TAK mengenali serangan nyata (recall ~0) akibat
   *distribution shift* pada fitur laju (`src_load`/`dst_load`) -> menegaskan perlunya kalibrasi
   domain untuk deteksi lintas-jaringan, bukan sekadar fitur/satuan yang benar.
7. **Pergeseran bergantung-kategori (biner -> multi-kelas)**: matriks $W_1$ per-kategori menunjukkan
   fitur pendorong shift berbeda antar kategori (BruteForce->`dst_load`, DDoS->`fwd_mean`,
   Botnet->`bwd_mean`), dan kategori sepadan bergeser berbeda antar-jaringan (CIC-DoS vs UNSW-DoS).
   Konsekuensinya: pemulihan 1% label cukup untuk biner, tetapi multi-kelas menuntut **penganggaran
   few-shot sadar-kelas** -> arah kerja mendatar (bukan klaim makalah ini).
8. **Biner vs Multi-class (dipisah agar mudah dipelajari)**: seluruh alur **BINER** (training,
   testing offline, SFM, diagnosis, testing AWS) di Bagian 3–7; lalu alur **MULTI-CLASS** di
   Bagian 8 (offline in-domain, XGBoost `multi:softprob`) & 8b (AWS). Multi-class: kategori bervolume
   tinggi (DDoS/DoS, Generic) terklasifikasi baik, kategori langka lebih sulit (class-imbalance);
   di AWS zero-shot tetap rendah karena distribution shift (sama seperti biner).

*Semua angka berasal dari eksperimen nyata dan dilaporkan apa adanya.*

## 11. Pembatasan Klaim Kausal *Distribution Shift* (untuk diskusi promotor)

> **Isu fundamental (promotor):** klaim *"gap disebabkan distribution shift, bukan feature
> insufficiency"* belum membuktikan **pemisahan kausal** penuh. *Joint training* HANYA
> membuktikan **fitur cukup** -- itu belum otomatis berarti distribution shift adalah
> **penyebab tunggal**. Masih ada faktor lain yang mungkin ikut berkontribusi.

**Lima faktor kandidat & status bukti nyata** (hanya notebook Paper 1: `01`-`10`, `20`-`25`, `30`):

| # | Faktor | Bukti nyata yang ADA | Terisolasi? |
|---|---|---|---|
| 1 | Feature insufficiency | Joint training MCC **0,912** (CIC) / **0,732** (UNSW) ~ in-domain (nb `05`) | **Ya - tersingkir** |
| 2 | Feature-extractor mismatch | Studi kasus `swin`/`dwin` + audit satuan (nb `02`); dinetralkan *by-design* via SFM | Dinetralkan, tak dikuantifikasi |
| 3 | Covariate/distribution shift | Wasserstein $W_1$ (nb `10`) + domain-classifier **~0,99** (nb `21`) | **Ya - terukur** ($P(X)$ marginal) |
| 4 | Label/attack-taxonomy shift | Per-kategori $W_1$ (nb `23`), transfer multiclass kelas-bersama kolaps (nb `24`) | Tidak - gejala tampak, kontribusi tak dipisah |
| 5 | Classifier limitation | Hanya XGBoost sbg detektor (RandomForest di nb `21` hanya *domain-classifier*) | Tidak - tak ada pembanding |

**Keputusan (Jalur A - reframe, tanpa eksperimen baru):** klaim di naskah diturunkan dari
*kausal-tunggal* menjadi **kausal-dominan + pengakuan jujur**:

- **Dibuktikan:** (a) fitur *cukup* (joint training menyingkirkan #1); (b) *covariate shift* besar
  yang **diukur langsung** (Wasserstein + domain-classifier ~0,99) -> *distribution shift* = penyebab **dominan**.
- **Diakui belum diisolasi:** taksonomi/label serangan ($P(Y)$ & $P(X\mid Y)$ per-kategori) dan
  kemungkinan *classifier ceiling* (XGBoost sengaja dikunci sbg *backbone*). Dekomposisi kausal penuh = *future work*.

**Bila promotor menuntut bukti isolasi lebih kuat (Jalur B, belum dikerjakan):** notebook baru
`26_causal_separation.ipynb` -> (i) replikasi kolaps->pulih pada detektor **non-XGBoost**
(RandomForest/MLP/LogReg) untuk menutup #5; (ii) dekomposisi *covariate* vs *label* shift pada
kelas bersama untuk menutup #4. Keduanya pakai data yang sudah ada, angka HARUS dari eksekusi nyata.

*Catatan: notebook `11`-`17` adalah Paper 2 (adversarial) - tidak relevan & tidak dicampur di sini.*